# Loading Libraries and Packages

In [63]:
import os, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image


# Setting the random seed for reproducibility
np.random.seed(42)

import warnings
warnings.filterwarnings("ignore")

if torch.backends.mps.is_available():
    print(f"MPS available. \nUsing MPS as runtime.")
    device = torch.device("mps")
    
elif torch.cuda.is_available():
    print(f"CUDA available. \nUsing CUDA as runtime.")
    device = torch.device("cuda")

else:
    print(f"No GPU available. \nUsing CPU runtime.")
    device = torch.device('cpu')

MPS available. 
Using MPS as runtime.


# Phase 2: Subspace & Clustering (Air Quality & PlantVillage)

## 2.5 PCA vs. Linear Autoencoders

### Load and Flatten Data

In [6]:
def flatten_data(X, window = 72, time_ahead = 24):
    
    X_flat = []
    curr_index =  window  # We do not include current time step. 
    while (curr_index+time_ahead) < X.shape[0]:
        X_flat.append(X[curr_index-window:curr_index].flatten())
        curr_index += 1
        
    X_flat = np.array(X_flat)
    
    return X_flat



def get_phase2_data(data_path, input_columns, time_steps = 24, sliding_window = 72, preprocess = True):
    # Creating Dataframe
    df = pd.read_csv(data_path)
    
    # Input Data
    X = np.array(df[input_columns].copy())
    
    # Train-Validation-Split Split (70-15-15)
    train_pct = 0.7
    val_pct = 0.15
    test_pct = 0.15
    
    
    train_index = int(X.shape[0]*train_pct)
    val_index = train_index + int(math.ceil(X.shape[0]*val_pct))
    
    X_train = X[:train_index].copy() # Training Data
    X_val = X[train_index:val_index].copy() # Validation Data
    X_test = X[val_index:].copy() # Test Data

    std_scaler = StandardScaler()

    # Pre-processing Data
    if preprocess:
        print("Standard Scaler applied on Data.")
        # --- Correct scaling ---
        X_train = std_scaler.fit_transform(X_train)
        X_val   = std_scaler.transform(X_val)
        X_test  = std_scaler.transform(X_test)

    print("Input Data Shape:", X.shape)
    
    print()
    print("#"*10, "Dimensions of Original Dataset","#"*10)
    print()
    
    print(f"Train Data Shape: {X_train.shape}")
    print(f"Validation Data Shape: {X_val.shape}")
    print(f"Test Data Shape: {X_test.shape}")
    
    print()
    print("#"*10, "Dimensions After Flattening","#"*10)
    print()
    X_train = flatten_data(X_train)
    X_val = flatten_data(X_val)
    X_test = flatten_data(X_test)
    
    print(f"Train Data Shape: {X_train.shape}")
    print(f"Validation Data Shape: {X_val.shape}")
    print(f"Test Data Shape: {X_test.shape}")
    
    return {
        'train': X_train,
        'val': X_val,
        'test': X_test,
        'std_scalers': std_scaler
    }

In [7]:
data_dir = "../Data"
data_path = os.path.join(data_dir, 'delhi_aqi.csv')

input_columns = ['co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']

# Getting and Flattening Data
phase2_data = get_phase2_data(data_path, input_columns, sliding_window = 72, time_steps = 24)
std_scaler_data = phase2_data['std_scalers'] # Needed to re-construct the original values

X_train = phase2_data['train']
X_val = phase2_data['val']
X_test = phase2_data['test']

X_train = torch.tensor(X_train, dtype = torch.float32).to(device)
X_val = torch.tensor(X_val, dtype = torch.float32).to(device)
X_test = torch.tensor(X_test, dtype = torch.float32).to(device)

Standard Scaler applied on Data.
Input Data Shape: (18776, 8)

########## Dimensions of Original Dataset ##########

Train Data Shape: (13143, 8)
Validation Data Shape: (2817, 8)
Test Data Shape: (2816, 8)

########## Dimensions After Flattening ##########

Train Data Shape: (13047, 576)
Validation Data Shape: (2721, 576)
Test Data Shape: (2720, 576)


### PCA via SVD

In [62]:
# Move to CPU for NumPy operations
X_train_np = X_train.detach().cpu().numpy()
X_val_np   = X_val.detach().cpu().numpy()
X_test_np  = X_test.detach().cpu().numpy()

print("Train shape:", X_train_np.shape)  # (N, 576)

Train shape: (13047, 576)


In [65]:
# Convert to numpy
X_train_np = X_train.detach().cpu().numpy()

# SVD
U, S, Vt = np.linalg.svd(X_train_np, full_matrices=False)
var_exp = (S**2) / np.sum(S**2)
var_exp

array([2.44326860e-01, 9.21017453e-02, 6.44645169e-02, 6.38765767e-02,
       5.49263023e-02, 5.46975322e-02, 4.58545648e-02, 2.84794066e-02,
       2.58272067e-02, 2.38920543e-02, 2.11313292e-02, 2.02344675e-02,
       1.36215482e-02, 1.30389668e-02, 9.37972497e-03, 9.09822341e-03,
       7.92234670e-03, 7.88715854e-03, 7.51779275e-03, 6.83078449e-03,
       6.59130048e-03, 5.96570969e-03, 5.93667058e-03, 5.88747673e-03,
       5.68243349e-03, 5.16868616e-03, 4.72515961e-03, 4.57456429e-03,
       4.46872646e-03, 4.43935534e-03, 4.28115902e-03, 4.15160833e-03,
       3.29052820e-03, 3.17363790e-03, 3.01172468e-03, 2.77801999e-03,
       2.61181011e-03, 2.56945938e-03, 2.41399766e-03, 2.33518868e-03,
       2.26582796e-03, 2.19826470e-03, 2.13008840e-03, 2.07706587e-03,
       2.02563126e-03, 1.98933179e-03, 1.98368821e-03, 1.97191373e-03,
       1.89449370e-03, 1.87014428e-03, 1.83590990e-03, 1.80107658e-03,
       1.75250554e-03, 1.71735638e-03, 1.69878162e-03, 1.63579558e-03,
      

In [11]:
# Convert to numpy
X_train_np = X_train.detach().cpu().numpy()

# SVD
U, S, Vt = np.linalg.svd(X_train_np, full_matrices=False)

# Explained variance
var_exp = (S**2) / np.sum(S**2)
cum_var = np.cumsum(var_exp)

# k for 95%
k = np.argmax(cum_var >= 0.95) + 1

print("Number of components (95% variance):", k)

Number of components (95% variance): 73


In [12]:
V_k = Vt[:k]

# Projection
X_pca = X_train_np @ V_k.T

# Reconstruction
X_pca_recon = X_pca @ V_k

# MSE
pca_mse = np.mean((X_train_np - X_pca_recon) ** 2)

print("PCA Reconstruction MSE:", pca_mse)

PCA Reconstruction MSE: 0.04987526


### Auto-Encoder

In [13]:
import torch.nn as nn

class LinearAE(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()
        self.encoder = nn.Linear(input_dim, bottleneck_dim, bias=False)
        self.decoder = nn.Linear(bottleneck_dim, input_dim, bias=False)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

In [14]:
input_dim = X_train.shape[1]

model = LinearAE(input_dim, k).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
dataset = TensorDataset(X_train)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

In [15]:
epochs = 200

for epoch in range(epochs):
    total_loss = 0
    
    for batch in loader:
        x = batch[0]
        
        optimizer.zero_grad()
        recon = model(x)
        loss = criterion(recon, x)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.6f}")

Epoch 10, Loss: 0.080529
Epoch 20, Loss: 0.056401
Epoch 30, Loss: 0.051653
Epoch 40, Loss: 0.050906
Epoch 50, Loss: 0.050912
Epoch 60, Loss: 0.050729
Epoch 70, Loss: 0.050638
Epoch 80, Loss: 0.051087
Epoch 90, Loss: 0.050589
Epoch 100, Loss: 0.050624
Epoch 110, Loss: 0.050574
Epoch 120, Loss: 0.050597
Epoch 130, Loss: 0.050546
Epoch 140, Loss: 0.050510
Epoch 150, Loss: 0.050690
Epoch 160, Loss: 0.050533
Epoch 170, Loss: 0.050499
Epoch 180, Loss: 0.050514
Epoch 190, Loss: 0.050553
Epoch 200, Loss: 0.050641


In [16]:
with torch.no_grad():
    X_train_recon_ae = model(X_train).cpu().numpy()

ae_mse = np.mean((X_train_np - X_train_recon_ae) ** 2)

print("Autoencoder Reconstruction MSE:", ae_mse)

Autoencoder Reconstruction MSE: 0.050143596


In [19]:
## Final Comparison

In [20]:
print("PCA MSE:", pca_mse)
print("AE  MSE:", ae_mse)

PCA MSE: 0.04987526
AE  MSE: 0.050143596


## 2.6 GMMLatent Clustering

### Dataset Class

In [33]:
#########  DATASET CLASS  #########

class PlantVillageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.labels_to_index = {}
        self.index_to_labels = {}
        self.samples = []
        self.labels = []   # needed for stratified split

        self.valid_extensions = (".jpg",".jpeg", ".png")
        
        # List of class names (folders in the root directory)
        class_names = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))])

        for labels_index,class_name in enumerate(class_names):
            class_dir = os.path.join(root_dir, class_name)
            self.labels_to_index[class_name] = labels_index
            self.index_to_labels[labels_index] = class_name

            for file_name in os.listdir(class_dir):
                if not file_name.lower().endswith(self.valid_extensions):
                    continue

                path = os.path.join(class_dir, file_name)
                if not os.path.isfile(path):
                    continue
                    
                self.samples.append((path, labels_index))
                self.labels.append(labels_index)
            
    def __len__(self):
        return len(self.samples)

    def num_classes(self):
        return len(self.labels_to_index)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

def sample_subset(dataset, num_samples=100, seed=42):
    random.seed(seed)
    indices = list(range(len(dataset)))
    sampled_indices = random.sample(indices, num_samples)
    return sampled_indices


#########  STRATIFIED SAMPLING  #########

def stratified_split(dataset, random_state=42):
    indices = list(range(len(dataset)))
    labels = dataset.labels

    # Step 1: Train (70) vs Temp (30)
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.3,
        stratify=labels,
        random_state=random_state
    )

    # Labels for temp split
    temp_labels = [labels[i] for i in temp_idx]

    # Step 2: Temp → Val (15) + Test (15)
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.5,
        stratify=temp_labels,
        random_state=random_state
    )

    return train_idx, val_idx, test_idx

### CNN Model Class

In [42]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2)
        )

    def forward(self, x):
        return self.block(x)

        
class CNNClassifier(nn.Module):
    def __init__(self, num_classes):
        
        super().__init__()
        self.num_classes = num_classes
        # Feature extractor
        self.features = nn.Sequential(
            ConvBlock(3, 32),    # 224 → 112
            ConvBlock(32, 64),   # 112 → 56
            ConvBlock(64, 128),  # 56 → 28
            ConvBlock(128, 256), # 28 → 14
        )

        # Adaptive pooling → fixed size
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)   # scalar output
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x  # shape: (batch,num_classes)

### Driver Code

In [46]:
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


root_dir= '../../Assignment 2/plantvillage dataset/color'
train_batch_size = 128
test_batch_size = 64
dataset = PlantVillageDataset(root_dir, transform=base_transform)
train_idx, val_idx, test_idx = stratified_split(dataset)
val_dataset   = Subset(dataset, val_idx)
val_loader = DataLoader(val_dataset, batch_size=test_batch_size, shuffle=False)

In [58]:
model_name = "2_6_1.pth"
model_path = f"../../Assignment 2/Models/{model_name}"

# Recreate model architecture
num_classes = dataset.num_classes()
model = CNNClassifier(num_classes= num_classes).to(device)

# Load saved weights
model.load_state_dict(torch.load(model_path, map_location=device))
print("Model Loaded Successfully!")

Model Loaded Successfully!


In [59]:
# Feature extractor (without classification head)
model.eval()

def extract_embeddings(model, loader, device):
    embeddings = []
    labels = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)

            # Forward pass until global pooling
            x = model.features(images)
            x = model.global_pool(x)        # (B, 256, 1, 1)
            x = x.view(x.size(0), -1)       # (B, 256)

            embeddings.append(x.cpu().numpy())
            labels.append(targets.numpy())

    X_embed = np.vstack(embeddings)
    y_true = np.concatenate(labels)

    return X_embed, y_true


X_embed, y_true = extract_embeddings(model, val_loader, device)

print("Embedding shape:", X_embed.shape)   # (N, 256)
print("Labels shape:", y_true.shape)

Embedding shape: (8146, 256)
Labels shape: (8146,)


In [60]:
from sklearn.mixture import GaussianMixture

num_clusters = num_classes  # same as number of disease classes

gmm = GaussianMixture(
    n_components=num_clusters,
    covariance_type='full',
    random_state=42
)

gmm.fit(X_embed)

,n_components,38
,covariance_type,'full'
,tol,0.001
,reg_covar,1e-06
,max_iter,100
,n_init,1
,init_params,'kmeans'
,weights_init,None
,means_init,None
,precisions_init,None
,random_state,42


In [61]:
from sklearn.metrics import adjusted_rand_score
y_pred = gmm.predict(X_embed)
ari = adjusted_rand_score(y_true, y_pred)
print("Adjusted Rand Index (ARI):", ari)

Adjusted Rand Index (ARI): 0.4658672845032758
